# Revelation Cross-Reference Analysis

This notebook summarizes cross-references to and from Revelation using the processed OpenBible.info cross-reference dataset.

Definitions used throughout:

- `rev_from`: rows where Revelation is the source book (`From Book == "Rev"`).
- `rev_to`: rows where Revelation is the target book (`To Verse start Book == "Rev"`).

In [1]:
from pathlib import Path
import sys

import pandas as pd
import plotly.express as px

cwd = Path.cwd().resolve()
if (cwd / "data").exists() and (cwd / "src").exists():
    PROJECT_DIR = cwd
elif cwd.name == "notebooks" and (cwd.parent / "data").exists():
    PROJECT_DIR = cwd.parent
elif (cwd / "revelation" / "data").exists():
    PROJECT_DIR = cwd / "revelation"
else:
    raise FileNotFoundError("Could not locate the revelation/data directory from the current working directory.")

SRC_DIR = PROJECT_DIR / "src"
DATA_DIR = PROJECT_DIR / "data"

if str(SRC_DIR) not in sys.path:
    sys.path.append(str(SRC_DIR))

from openbible_data import get_book_order, load_cross_references
from revelation_analysis import (
    count_references_to_revelation,
    get_rev_from,
    get_rev_to,
    rank_nt_books_by_reference_method,
    summarize_revelation_counts,
    summarize_revelation_testaments,
)

In [2]:
df = load_cross_references(DATA_DIR / "cross_references_revelation_public.csv")
book_order = get_book_order(df)

rev_from = get_rev_from(df)
rev_to = get_rev_to(df)

df.shape

(344799, 18)

## Core Revelation Counts

These summary rows mirror the directional language used in the Revelation Graphic Explainer: references **to** Revelation are incoming references; references **from** Revelation are outgoing references.

In [3]:
summary_counts = summarize_revelation_counts(df)
summary_counts

,Metric,Count,Definition
0,Cross-references to Revelation,9943,Rows where To Verse start Book is Rev
1,Cross-references from Revelation,6495,Rows where From Book is Rev
2,Internal Revelation references,2205,Rows where both From Book and To Verse start B...
3,Cross-references to Revelation from other books,7738,Rows where To Verse start Book is Rev and From...


In [4]:
testament_summary = summarize_revelation_testaments(df)
testament_summary

,Direction,Testament field,Old Testament,New Testament,Total
0,To Revelation,From Book Testament,4777,5166,9943
1,From Revelation,To Book Testament,2500,3995,6495


## New Testament Ranking Methods

The phrase "most cross-referenced" can mean more than one thing. The table below keeps three methods side by side:

- `Incoming`: references to the book.
- `Outgoing`: references from the book.
- `Combined`: incoming plus outgoing.

In [5]:
nt_rankings = rank_nt_books_by_reference_method(df)
nt_rankings.sort_values("Incoming rank").head(10)

,Book,Book number,Outgoing,Incoming,Combined,Outgoing rank,Incoming rank,Combined rank
4777,Matt,40,13833,14785,28618,1,1,1
5998,Acts,44,13319,13761,27080,2,2,2
5666,John,43,11797,12856,24653,4,3,3
5343,Luke,42,12371,11803,24174,3,4,4
7738,Rev,66,6495,9943,16438,6,5,6
6243,Rom,45,7750,9052,16802,5,6,5
7306,Heb,58,5562,6900,12462,9,7,7
6446,1Cor,46,5717,6316,12033,8,8,8
6588,2Cor,47,4253,5035,9288,10,9,10
5216,Mark,41,6468,4529,10997,7,10,9


In [6]:
nt_rankings.loc[nt_rankings["Book"] == "Rev"]

,Book,Book number,Outgoing,Incoming,Combined,Outgoing rank,Incoming rank,Combined rank
7738,Rev,66,6495,9943,16438,6,5,6


## Cross-References to Revelation by Source Book

This chart counts rows where Revelation is the target book. The large final bar is internal Revelation-to-Revelation cross-references.

In [7]:
to_revelation_counts = count_references_to_revelation(df, include_revelation=True)

to_revelation_counts["Testament"] = to_revelation_counts["Testament"].replace({
    "Old": "OT",
    "New": "NT",
})

to_revelation_counts.head()

,Book,Book number,Testament,Count
29,Gen,1,OT,139
25,Exod,2,OT,201
45,Lev,3,OT,96
53,Num,4,OT,99
21,Deut,5,OT,103


In [8]:
fig = px.bar(
    to_revelation_counts,
    x="Book",
    y="Count",
    color="Testament",
    title="Cross-References to Revelation",
    labels={"Book": "From Book", "Count": "Count", "Testament": "Testament"},
    category_orders={"Book": book_order},
    color_discrete_map={"OT": "#636EFA", "NT": "#EF553B"},
)

fig.update_layout(width=1200, height=560)
fig.update_xaxes(tickangle=270)
fig.show()

## Cross-References to Revelation Excluding Internal References

For some teaching contexts it is useful to remove Revelation-to-Revelation rows and focus only on references from other books to Revelation.

In [9]:
to_revelation_external_counts = count_references_to_revelation(df, include_revelation=False)
to_revelation_external_counts["Testament"] = to_revelation_external_counts["Testament"].replace({
    "Old": "OT",
    "New": "NT",
})

to_revelation_external_counts.tail()

,Book,Book number,Testament,Count
12,2Pet,61,NT,49
2,1John,62,NT,74
10,2John,63,NT,4
16,3John,64,NT,2
42,Jude,65,NT,24


In [10]:
fig = px.bar(
    to_revelation_external_counts,
    x="Book",
    y="Count",
    color="Testament",
    title="Cross-References to Revelation from Other Books",
    labels={"Book": "From Book", "Count": "Count", "Testament": "Testament"},
    category_orders={"Book": book_order},
    color_discrete_map={"OT": "#636EFA", "NT": "#EF553B"},
)

fig.update_layout(width=1200, height=560)
fig.update_xaxes(tickangle=270)
fig.show()